In [14]:
import pandas as pd
import numpy as np

def preprocess_air_quality_data(raw_filepath, processed_filepath, metadata_filepath):
    print("Loading raw dataset...")
    df = pd.read_csv(raw_filepath)
    
    # 1. Standardize the primary Datetime column
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # 2. Extract Static Node Metadata (Spatial Graph Info)
    print("Extracting Node Metadata (Lat/Lon)...")
    node_meta = df[['station', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
    node_meta.to_csv(metadata_filepath, index=False)
    
    # 3. Drop redundant columns (neural networks only want pure numbers and no duplicated data)
    cols_to_drop = [
        'date', 'year', 'month', 'day', 'day_of_week', 'season', 
        'city', 'aqi_category', 'latitude', 'longitude'
    ]
    df = df.drop(columns=cols_to_drop)

    # 4. Enforce Strict 6-Hour Intervals (Fixing the 23:00 vs 00:00 drift)
    print("Resampling time-series to strict 6-hour intervals per station...")
    dfs_resampled = []
    stations = df['station'].unique()

    for st in stations:
        # Filter for one station at a time
        df_loc = df[df['station'] == st].copy()
        df_loc.set_index('datetime', inplace=True)
        df_loc = df_loc.drop(columns=['station'])
        
        # Resample strictly to 6H ('00:00', '06:00', '12:00', '18:00')
        # This will create missing values for '00:00' since the raw data uses '23:00'
        df_loc = df_loc.resample('6h').mean(numeric_only=True)
        
        # Linearly interpolate to logically fill the missing 00:00 values based on 23:00 & 06:00
        df_loc = df_loc.interpolate(method='linear')
        
        # Catch any trailing edge NaNs
        df_loc = df_loc.ffill().bfill() 
        
        # Restore station ID
        df_loc['station'] = st
        dfs_resampled.append(df_loc)

    # Recombine all the stations
    df_processed = pd.concat(dfs_resampled)

    # 5. Add Cyclical Temporal Features (Time-of-day awareness)
    print("Encoding cyclical time features...")
    hour = df_processed.index.hour
    
    # Sine/Cosine encoding ensures the AI knows that 18:00 and 00:00 are close in a cycle
    df_processed['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df_processed['hour_cos'] = np.cos(2 * np.pi * hour / 24)

    # Reset index to save cleanly
    df_processed = df_processed.reset_index()

    # 6. Save the final processed dataset
    df_processed.to_csv(processed_filepath, index=False)
    
    print(f"\n✅ Processing Complete!")
    print(f"-> Time-Series Data (Inputs X) saved to: {processed_filepath} | Shape: {df_processed.shape}")
    print(f"-> Spatial Graph Data (Nodes V) saved to: {metadata_filepath} | Nodes: {len(node_meta)}")

# Execution Block
if __name__ == "__main__":
    preprocess_air_quality_data(
        raw_filepath='../data/raw/delhi_data.csv',
        processed_filepath='../data/processed/processed_delhi_data.csv', 
        metadata_filepath='../data/processed/node_metadata.csv'
    )

Loading raw dataset...
Extracting Node Metadata (Lat/Lon)...
Resampling time-series to strict 6-hour intervals per station...
Encoding cyclical time features...

✅ Processing Complete!
-> Time-Series Data (Inputs X) saved to: ../data/processed/processed_delhi_data.csv | Shape: (201641, 17)
-> Spatial Graph Data (Nodes V) saved to: ../data/processed/node_metadata.csv | Nodes: 23


In [2]:
import pandas as pd

In [8]:
df_poll.head()

,date,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2020-11-25 01:00:00,2616.88,2.18,70.60,13.59,38.62,364.61,411.73,28.63
1,2020-11-25 02:00:00,3631.59,23.25,89.11,0.33,54.36,420.96,486.21,41.04
2,2020-11-25 03:00:00,4539.49,52.75,100.08,1.11,68.67,463.68,541.95,49.14
3,2020-11-25 04:00:00,4539.49,50.96,111.04,6.44,78.20,454.81,534.00,48.13
4,2020-11-25 05:00:00,4379.27,42.92,117.90,17.17,87.74,448.14,529.19,46.61
